# LightHit: один фотон и точечный приёмник

Сначала решаем монохромную задачу в однородной бесконечной среде. Источник — мгновенная вспышка в заданном направлении. Приёмник изотропен и имеет эффективность 100%.

**Результат нормирован на единицу эффективной площади:** заряд в м⁻² на испущенный фотон, временная плотность в м⁻²·нс⁻¹. Приписывать точке конечную геометрическую площадь не будем. Для малого детектора с площадью $A$ математическое ожидание сигнала равно $A Q$.

Ни приватная вода, ни характеристики реального ОМ в этом ноутбуке по умолчанию не используются. Все численные параметры среды ниже — тестовые.

Запуск: из корня установленного проекта `python -m jupyter lab notebooks/01_point_green.ipynb`, затем **Run → Run All Cells**. Код физики находится в `src/lighthit`, а не дублируется в ячейках. Подробный вывод: `docs/point-green.qmd`.

In [ ]:
%matplotlib inline
from dataclasses import asdict, replace
from pathlib import Path
from time import perf_counter
import json
import numpy as np
import matplotlib.pyplot as plt
from scipy.special import eval_legendre

import lighthit
from lighthit import Medium, synthetic_medium, PointGreenSolver, SolverSettings
from lighthit.angular import angular_components, dense_finite_rank_reference
from lighthit.single import hg_phase

print("LightHit:", lighthit.__version__)
print("Модуль:", lighthit.__file__)
root = Path.cwd()
if not (root / "pyproject.toml").exists() and (root.parent / "pyproject.toml").exists():
    root = root.parent
output = root / ".build" / "notebook"
output.mkdir(parents=True, exist_ok=True)

## 1. Параметры среды

$\mu_a$ и $\mu_s$ — вероятности поглощения и рассеяния на единицу пути, м⁻¹. $v=c_0/n_g$ — групповая скорость. HG-параметр $g$ задаёт средний косинус однократного рассеяния. Фазовая функция нормирована по полному телесному углу:

$$p(x)=\frac{1-g^2}{4\pi(1+g^2-2gx)^{3/2}},\qquad x=\cos\theta.$$

$1/[\mu_s(1-g)]$ — характерная длина потери памяти о первоначальном направлении, без добавления поглощения.

In [ ]:
medium = synthetic_medium()
settings = SolverSettings()  # L=32, J=160, k_max=6 м^-1
print(json.dumps(asdict(medium), indent=2, ensure_ascii=False))
print("v [м/нс] =", medium.speed_m_per_ns)
print("Длина поглощения [м] =", 1 / medium.absorption_per_m)
print("Длина рассеяния [м] =", 1 / medium.scattering_per_m)
print("Длина потери направления [м] =", 1 / (medium.scattering_per_m * (1 - medium.g)))
print("Численные настройки:", asdict(settings))

In [ ]:
mu = np.linspace(-1, 1, 600)
ell = np.arange(settings.scattering_degree + 1)
finite_phase = np.sum(
    ((2*ell+1) * medium.g**ell / (4*np.pi))[:, None]
    * eval_legendre(ell[:, None], mu), axis=0
)
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.semilogy(mu, hg_phase(mu, medium.g), label="HG")
ax.semilogy(mu, finite_phase, linestyle="--", label=f"HG до L={settings.scattering_degree}")
ax.set(xlabel="cos(theta)", ylabel="p [sr^-1]", title="Фазовая функция рассеяния")
ax.legend(); fig.tight_layout(); plt.show()

## 2. Геометрия направленной вспышки

Источник находится в начале координат и испускает **один** фотон вдоль $\mathbf s_0=(0,0,1)$. Вектор $\mathbf r=\mathbf R_d-\mathbf x$ направлен от источника к приёмнику. Достаточно его длины $r$ и косинуса $\nu=\widehat{\mathbf r}\cdot\mathbf s_0$.

В примере $r=20$ м и угол равен 60°. Баллистический луч не проходит через приёмник, поэтому $Q^{(0)}=0$. Все зарегистрированные фотоны должны рассеяться. На точном прямом луче идеальная направленная вспышка даёт сингулярный отклик; API отклоняет такую геометрию.

In [ ]:
direction = np.array([0., 0., 1.])
r = 20.0
angle_deg = 60.0
angle = np.deg2rad(angle_deg)
displacement = r * np.array([np.sin(angle), 0.0, np.cos(angle)])
front = r / medium.speed_m_per_ns
print("r [м] =", r, "; cos угла =", displacement @ direction / r)
print("Самое раннее возможное время r/v [нс] =", front)
fig, ax = plt.subplots(figsize=(6, 5))
ax.plot([0, displacement[0]], [0, displacement[2]], linestyle=":", label="источник–приёмник")
ax.plot([0, 0], [0, 25], label="начальный луч")
ax.scatter([0, displacement[0]], [0, displacement[2]])
ax.annotate("источник", (0, 0), xytext=(5, 8), textcoords="offset points")
ax.annotate("приёмник", (displacement[0], displacement[2]), xytext=(5, 8), textcoords="offset points")
ax.set(xlabel="x [м]", ylabel="z [м]", title="Геометрия")
ax.set_aspect("equal"); ax.legend(); fig.tight_layout(); plt.show()

## 3. Уравнение в Фурье-пространстве

Соглашение: $\widehat I=\int I e^{-i\mathbf k\cdot\mathbf x+i\omega t}\,d^3x\,dt$.

$$D(\mathbf s)=d_0+ik\mu,\quad d_0=\mu_t-i\omega/v,\quad \mu=\widehat{\mathbf k}\cdot\mathbf s.$$

Сопряжённая правая часть для изотропного детектора равна 1. В коде решается уравнение для комплексно-сопряжённого сопряжённого отклика $h=\psi^*$:

$$(D-V)h=1.$$

Остаётся только азимутальный блок $m=0$. В нормированных полиномах $p_l=\sqrt{(2l+1)/2}P_l$ коэффициенты удовлетворяют трёхдиагональной системе:

$$ (d_0-\gamma_l)h_l+ik a_l h_{l-1}+ik a_{l+1}h_{l+1}=\sqrt2\delta_{l0},
\qquad a_l=\frac{l}{\sqrt{4l^2-1}}.$$

$\gamma_l=\mu_sg^l$ при $l\le L$, а выше $L$ — ноль. Бесконечный свободный хвост исключается через точное отношение свободных моментов $h_{L+1}/h_L=b_{L+1}/b_L$.

В итоге решаем две системы размера **$L+1$**, затем продолжаем свободный хвост до требуемой степени пространственного обращения $J$. Условие $h_{L+1}=0$ не используется. Поле $I$ не строится.

### Независимая проверка матрицы

Ниже для одной спектральной точки сравниваем прогонку с плотным решателем. Второй способ строит $B_{ij}=\int p_i p_j/D\,d\mu$ независимой угловой квадратурой. Сравнение не проверяет пространственное и временное обращение; они проверяются отдельно.

In [ ]:
k_test = 0.2  # м^-1
omega_test = 0.1  # рад/нс
L_test, J_test = 8, 16
free, one, multiple = angular_components(
    np.array([k_test]), omega_test, medium, L_test, J_test
)
reference = dense_finite_rank_reference(k_test, omega_test, medium, L_test, J_test)
full = (free + one + multiple)[0]
print("Максимальная абсолютная разность коэффициентов:", np.max(abs(full - reference)))
print("Первые четыре полных коэффициента:", full[:4])

## 4. Все порядки и пространственное обращение

Внутри решателя $h^{(0)}=b$, $A_0h^{(1)}=\Gamma b$, $A_Lh^{(\ge2)}=\Gamma h^{(1)}$. Последняя система суммирует все числа столкновений $2,3,\ldots$.

Угловой интеграл по направлению $\mathbf k$ выполнен аналитически:

$$\widehat K^{(\ge2)}(r,\nu,\omega)=\frac1{2\pi^2}
\sum_{l=0}^{J}i^l\sqrt{\frac{2l+1}{2}}P_l(\nu)
\int_0^\infty k^2j_l(kr)h_l^{(\ge2)}(k,\omega)\,dk.$$

$j_l$ — сферическая функция Бесселя. Численной остаётся одномерная радиальная квадратура по $k$.

Нулевой порядок и первый порядок считаются отдельно в координатном пространстве. Итог: $K^{(0)}+K^{(1)}_{\rm HG}+K^{(\ge2)}_L$. Первый порядок использует полную HG-функцию, многократная часть — её коэффициенты до $L$. Два независимых угловых разрешения: $L$ для рассеяния, $J$ для пространственного обращения.

In [ ]:
omega = np.linspace(0.0, 1.2, 241)
solver = PointGreenSolver(medium, settings)
started = perf_counter()
result = solver.solve(omega, displacement, direction=direction)
print("Полное время вызова [с]:", perf_counter() - started)
print(json.dumps(result.timings_s, indent=2))
result.save(output / "directed-spectrum.npz")

## 5. Заряд и спектр

Заряд равен $Q=\widehat K(0)$, м⁻². Он вычисляется независимо от конечного временного окна.

Частотный спектр ниже умножен на $e^{-i\omega r/v}$ только для изображения: известная фаза светового фронта скрывает форму более медленной амплитуды. Сохранённый спектр не модифицируется.

In [ ]:
names = ["0", "1 (полная HG)", ">=2 (HG до L)"]
for name, value in zip(names, result.components[0, 0].real):
    print(f"Q[{name}] = {value:.10e} м^-2")
print(f"Q[весь сигнал] = {result.charge_per_m2[0]:.10e} м^-2")
print("Доля >=2:", result.components[0, 0, 2].real / result.charge_per_m2[0])

demodulated = result.spectrum[:, 0] * np.exp(-1j * omega * front)
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(omega, demodulated.real, label="Re")
ax.plot(omega, demodulated.imag, label="Im")
ax.set(xlabel="omega [рад/нс]", ylabel="Спектр [м^-2]", title="Спектр без фазы r/v")
ax.legend(); fig.tight_layout(); plt.show()

## 6. Временные бины

Для многократной части используем

$$N_b=\frac1\pi\Re\sum_jw_j\widehat K(\omega_j)
 e^{-\omega_j^2\sigma^2/2}
\Delta t_b\operatorname{sinc}(\omega_j\Delta t_b/2)e^{-i\omega_jt_b}.$$

$w_j$ — веса частотной квадратуры, $t_b$ и $\Delta t_b$ — центр и ширина бина. Первые два порядка интегрируются непосредственно по времени. При $\sigma=0$ аппаратного размытия нет; при $\sigma=3$ нс вводится отдельный гауссовский readout.

Конечные $k$ и $\omega$ вызывают осцилляции у фронта. Мы сохраняем их, включая отрицательные бины. Гауссовский readout после расчёта может заметно уменьшить их. Он не доказывает точность неразмытого фронта.

In [ ]:
edges = np.arange(front - 30, front + 651, 2.0)
raw = result.readout(edges, sigma_ns=0.0)
measured = result.readout(edges, sigma_ns=3.0)
print("Время readout без размытия [с]:", raw.elapsed_s)
print("Время readout sigma=3 нс [с]:", measured.elapsed_s)
print("Сумма бинов / заряд:", raw.diagnostics["window_charge_fraction"])
print("Отрицательная масса без размытия / заряд:", raw.diagnostics["negative_mass_fraction"])
print("Модуль сигнала до фронта без размытия / заряд:", raw.diagnostics["prefront_absolute_mass_fraction"])
print("Отрицательная масса sigma=3 / заряд:", measured.diagnostics["negative_mass_fraction"])
print("Период частотной суммы [нс]:", 2 * np.pi / (omega[1] - omega[0]))

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.8))
for p, name in enumerate(names):
    ax.plot(raw.centers_ns, raw.components[0, :, p] / np.diff(edges), label=name)
ax.plot(raw.centers_ns, raw.rate_per_m2_ns[0], label="весь сигнал", linewidth=2)
ax.axvline(front, linestyle=":", label="r/v")
ax.set(xlabel="t [нс]", ylabel="Отклик [м^-2 нс^-1]", title="Без аппаратного размытия")
ax.legend(); fig.tight_layout(); plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.8))
for p, name in enumerate(names):
    ax.plot(measured.centers_ns, measured.components[0, :, p] / np.diff(edges), label=name)
ax.plot(measured.centers_ns, measured.rate_per_m2_ns[0], label="весь сигнал", linewidth=2)
ax.axvline(front, linestyle=":", label="r/v")
ax.set(xlabel="t [нс]", ylabel="Отклик [м^-2 нс^-1]", title="Гауссовский readout: sigma=3 нс")
ax.legend(); fig.tight_layout(); plt.show()

## 7. Численное уточнение

Сравним `quick` с основной настройкой. Это измерение изменения результата при конкретном уточнении, не универсальная граница ошибки. Для строгого исследования меняйте $L$, $J$, cutoff/панели $k$, полосу и шаг $\omega$ по отдельности.

Метрика временного различия — сумма абсолютных разностей бинов, делённая на полный заряд основного расчёта. Нормировку профиля не подгоняем.

In [ ]:
quick = PointGreenSolver(medium, SolverSettings.quick()).solve(omega, displacement, direction=direction)
quick_readout = quick.readout(edges, sigma_ns=3.0)
q = result.charge_per_m2[0]
print("Время quick [с]:", quick.timings_s["total"])
print("Разность зарядов / Q:", abs(quick.charge_per_m2[0] - q) / q)
print("L1 разность бинов sigma=3 / Q:", np.abs(quick_readout.values_per_m2 - measured.values_per_m2).sum() / q)

# Более дорогой контроль включается явно.
RUN_REFINED = False
if RUN_REFINED:
    refined = PointGreenSolver(medium, SolverSettings.refined()).solve(omega, displacement, direction=direction)
    refined_profile = refined.readout(edges, sigma_ns=3.0)
    print("refined Q [м^-2]:", refined.charge_per_m2)
    print("L1 main-refined / Q_ref:", np.abs(refined_profile.values_per_m2 - measured.values_per_m2).sum() / refined.charge_per_m2[0])

## 8. Несколько направлений и поворот всей геометрии

При фиксированных расстоянии и направлении испускания изменение угла на приёмник меняет отклик. Следующий вызов считает сразу четыре угла: угловая транспортная задача общая для них.

Совместный поворот источника и приёмника не меняет ответ. Поворот только направления испускания при фиксированном приёмнике в общем случае его меняет.

In [ ]:
angles = np.array([30., 60., 90., 120.])
a = np.deg2rad(angles)
positions = r * np.stack([np.sin(a), np.zeros_like(a), np.cos(a)], axis=1)
charges = solver.solve([0.0], positions, direction=direction)
print("Время четырёх зарядов [с]:", charges.timings_s["total"])
for angle_value, value in zip(angles, charges.charge_per_m2):
    print(f"Угол {angle_value:g}°: {value:.8e} м^-2")
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(angles, charges.charge_per_m2, marker="o")
ax.set(xlabel="Угол [градусы]", ylabel="Q [м^-2]", title="Зависимость от первоначального направления")
fig.tight_layout(); plt.show()

Qrot = np.array([[0., 0., 1.], [1., 0., 0.], [0., 1., 0.]])
rotated = solver.solve([0.0], Qrot @ displacement, direction=Qrot @ direction)
print("Относительное изменение при общем повороте:", abs(rotated.charge_per_m2[0] - q) / q)

## 9. Изотропная вспышка как контроль

`direction=None` задаёт равномерное испускание одного фотона на полной сфере, а не ещё один направленный луч. Появляется прямой импульс с зарядом $e^{-\mu_t r}/(4\pi r^2)$. В пространственном обращении нужен только монополь источника, поэтому этот расчёт существенно дешевле направленного.

При $g=0$ полный спектральный монополь дополнительно проверяется точной формулой
$A/(1-\mu_sA)$, где $A=\arctan(k/d_0)/k$. Эта проверка и независимое пространственное обращение находятся в `tests/`.

In [ ]:
isotropic = solver.solve(omega, displacement, direction=None)
iso_profile = isotropic.readout(edges, sigma_ns=3.0)
print("Изотропный заряд [м^-2]:", isotropic.charge_per_m2)
print("Прямой заряд (число из спектра):", isotropic.components[0, 0, 0].real)
print("Прямой заряд (формула):", np.exp(-medium.extinction_per_m * r) / (4*np.pi*r*r))
print("Время изотропного расчёта [с]:", isotropic.timings_s["total"])
fig, ax = plt.subplots(figsize=(8, 4.8))
for p, name in enumerate(names):
    ax.plot(iso_profile.centers_ns, iso_profile.components[0, :, p] / np.diff(edges), label=name)
ax.plot(iso_profile.centers_ns, iso_profile.rate_per_m2_ns[0], label="весь сигнал", linewidth=2)
ax.set(xlabel="t [нс]", ylabel="Отклик [м^-2 нс^-1]", title="Изотропная вспышка, sigma=3 нс")
ax.legend(); fig.tight_layout(); plt.show()

## 10. Подключение приватной воды — только локально

Код адаптера сохранён, приватные данные не входят в LightHit. Его интерфейс проверен на искусственном двойнике. С реальным приватным пакетом этот ноутбук здесь не выполнялся.

Для локального расчёта замените создание `medium` на:

```python
from lighthit.providers import load_bgvd_water
medium = load_bgvd_water(
    "/полный/локальный/путь/к/bgvd-model",
    wavelength_nm=450.0,
    g=0.9,
)
```

Потом пересоздайте `solver`. HG-параметр задаётся явно. Для новой воды заново исследуйте сходимость; пример $g=0.7$ не устанавливает рабочую степень для $g=0.9$.

**Не коммитьте выводы ноутбука с приватными параметрами или сигналами.** Сохраняйте результаты в `.build/`; перед добавлением исходного ноутбука в Git выполните `python scripts/strip_notebook_outputs.py notebooks/01_point_green.ipynb`.

## Что проверено этим примером

Получены все порядки рассеяния в объявленной численной схеме, спектр, заряд и временные бины. Показаны отдельные затраты на угловое решение, пространственное обращение и readout; проверены изменение угла и общий поворот.

Здесь ещё нет конечного ОМ, приватной калибровки, треков, ливней и транспортного кэша между вызовами. Неразмытый фронт требует более строгого контроля, чем сглаженный профиль. Отрицательные бины не удаляются.

Определения специальных функций: [NIST DLMF](https://dlmf.nist.gov/14). Радиальное обращение: [SciPy spherical_jn](https://docs.scipy.org/doc/scipy/reference/generated/scipy.special.spherical_jn.html). Полный текст вывода находится в `docs/point-green.qmd`.